# 07 - Elasticity Signal Model (EfficientNet-B0)

Train a Frangi wrinkle quantifier for the **elasticity** skin signal.

## Datasets
- **UTKFace** - age labels as elasticity proxy (age inversely correlates with elasticity)
- **FFHQ** (70k faces) - diverse age distribution for robust training
- **AgeDB** - age-annotated face images in the wild

## Features
- Frangi filter bank for tubular/wrinkle structure detection
- Forehead and crow's feet ROI extraction via face landmarks
- Landmark geometry ratios (distances between key facial points)

## Architecture
- Backbone: EfficientNet-B0 (pretrained ImageNet)
- Feature fusion: CNN features + Frangi filter features + landmark geometry
- Regression head for elasticity score [0-100]

## Output
- ONNX model for backend inference

In [ ]:
# Install dependencies (Colab)
!pip install -q torch torchvision timm albumentations onnx onnxruntime opencv-python scikit-image mediapipe

In [ ]:
import os
import json
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from skimage.filters import frangi
from tqdm import tqdm
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## Feature Extraction

Frangi filter detects tubular structures (wrinkles). ROI extraction focuses on
forehead and crow's feet regions where wrinkles are most indicative of elasticity.

In [ ]:
def extract_frangi_features(gray: np.ndarray) -> dict:
    """Apply Frangi filter bank to detect wrinkle-like tubular structures."""
    frangi_resp = frangi(
        gray.astype(np.float64) / 255.0,
        sigmas=range(1, 5),
        alpha=0.5,
        beta=0.5,
        gamma=15,
        black_ridges=True,
    )
    wrinkle_density = (frangi_resp > 0.01).mean()
    wrinkle_intensity = frangi_resp[frangi_resp > 0.01].mean() if wrinkle_density > 0 else 0.0
    return {
        "response": frangi_resp,
        "density": float(wrinkle_density),
        "intensity": float(wrinkle_intensity),
        "max_response": float(frangi_resp.max()),
    }


def extract_roi_features(gray: np.ndarray, landmarks: dict) -> np.ndarray:
    """Extract Frangi features from forehead and crow's feet ROIs.

    landmarks: dict with 'forehead' (x, y, w, h) and 'left_eye', 'right_eye' bounding boxes.
    """
    features = []
    for roi_name in ["forehead", "left_crows_feet", "right_crows_feet"]:
        if roi_name in landmarks:
            x, y, w, h = landmarks[roi_name]
            roi = gray[y:y+h, x:x+w]
            if roi.size > 0:
                f = extract_frangi_features(roi)
                features.extend([f["density"], f["intensity"], f["max_response"]])
            else:
                features.extend([0.0, 0.0, 0.0])
        else:
            features.extend([0.0, 0.0, 0.0])
    return np.array(features, dtype=np.float32)


def extract_landmark_geometry(landmarks_2d: np.ndarray) -> np.ndarray:
    """Compute geometric ratios from face landmarks indicating skin laxity.

    landmarks_2d: (N, 2) array of normalized landmark coordinates.
    """
    if landmarks_2d is None or len(landmarks_2d) < 68:
        return np.zeros(5, dtype=np.float32)

    # Key distances (using 68-point landmark indices)
    brow_eye_dist = np.linalg.norm(landmarks_2d[19] - landmarks_2d[37])  # brow to eye
    eye_width = np.linalg.norm(landmarks_2d[36] - landmarks_2d[39])      # eye width
    jaw_width = np.linalg.norm(landmarks_2d[3] - landmarks_2d[13])       # jaw width
    face_height = np.linalg.norm(landmarks_2d[8] - landmarks_2d[27])     # chin to nose bridge
    mouth_width = np.linalg.norm(landmarks_2d[48] - landmarks_2d[54])    # mouth width

    # Ratios that change with skin laxity
    return np.array([
        brow_eye_dist / (eye_width + 1e-6),
        jaw_width / (face_height + 1e-6),
        mouth_width / (jaw_width + 1e-6),
        face_height / (jaw_width + 1e-6),
        brow_eye_dist / (face_height + 1e-6),
    ], dtype=np.float32)


HANDCRAFTED_DIM = 9 + 5  # 3 ROIs x 3 features + 5 geometry ratios = 14
print(f"Handcrafted feature dimension: {HANDCRAFTED_DIM}")

## Dataset

Expects face images with age labels. Elasticity score is derived from age:
younger faces have higher elasticity scores.

In [ ]:
def age_to_elasticity(age: int) -> float:
    """Convert age to elasticity score [0-100]. Younger = higher elasticity."""
    if age <= 20:
        return 95.0
    elif age >= 80:
        return 15.0
    else:
        return 95.0 - (age - 20) * (80.0 / 60.0)


class ElasticityDataset(Dataset):
    """Dataset for elasticity signal training using age-labeled face images."""

    def __init__(self, image_dir: str, annotations_path: str, transform=None):
        self.image_dir = image_dir
        with open(annotations_path) as f:
            self.annotations = json.load(f)
        self.image_ids = list(self.annotations.keys())
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        ann = self.annotations[img_id]

        image = cv2.imread(os.path.join(self.image_dir, img_id))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_resized = cv2.resize(image, (224, 224))
        gray = cv2.cvtColor(image_resized, cv2.COLOR_RGB2GRAY)

        # Extract handcrafted features
        landmarks = ann.get("landmarks", {})
        roi_feat = extract_roi_features(gray, landmarks)
        landmarks_2d = np.array(ann["landmarks_2d"]) if "landmarks_2d" in ann else None
        geom_feat = extract_landmark_geometry(landmarks_2d)
        handcrafted = np.concatenate([roi_feat, geom_feat])

        if self.transform:
            augmented = self.transform(image=image_resized)
            image_tensor = augmented["image"]
        else:
            image_tensor = torch.from_numpy(image_resized.transpose(2, 0, 1)).float() / 255.0

        # Elasticity score from age or direct annotation
        if "elasticity_score" in ann:
            score = ann["elasticity_score"]
        else:
            score = age_to_elasticity(ann["age"])
        label = torch.tensor([score], dtype=torch.float32)

        return image_tensor, torch.from_numpy(handcrafted), label


train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

## Model Definition

EfficientNet-B0 backbone fused with Frangi/landmark handcrafted features.

In [ ]:
class ElasticityModel(nn.Module):
    """Elasticity signal model with EfficientNet-B0 + Frangi/landmark feature fusion."""

    def __init__(self, handcrafted_dim: int = 14, pretrained: bool = True):
        super().__init__()
        self.backbone = timm.create_model("efficientnet_b0", pretrained=pretrained, num_classes=0)
        cnn_dim = self.backbone.num_features  # 1280

        self.handcrafted_fc = nn.Sequential(
            nn.Linear(handcrafted_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.head = nn.Sequential(
            nn.Linear(cnn_dim + 32, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
        )

    def forward(self, image, handcrafted):
        cnn_feat = self.backbone(image)
        hc_feat = self.handcrafted_fc(handcrafted)
        fused = torch.cat([cnn_feat, hc_feat], dim=-1)
        return self.head(fused)


model = ElasticityModel(handcrafted_dim=HANDCRAFTED_DIM, pretrained=True).to(DEVICE)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Training Loop

In [ ]:
IMAGE_DIR = "./data/elasticity/images"
ANNOTATIONS_PATH = "./data/elasticity/annotations.json"

NUM_EPOCHS = 30
BATCH_SIZE = 32
LR = 1e-4


def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for images, handcrafted, labels in tqdm(loader, desc="Training"):
        images = images.to(DEVICE)
        handcrafted = handcrafted.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        preds = model(images, handcrafted)
        loss = nn.functional.smooth_l1_loss(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, handcrafted, labels in loader:
        images, handcrafted = images.to(DEVICE), handcrafted.to(DEVICE)
        preds = model(images, handcrafted)
        all_preds.append(preds.cpu())
        all_labels.append(labels)
    preds = torch.cat(all_preds).squeeze()
    labels = torch.cat(all_labels).squeeze()
    mae = torch.abs(preds - labels).mean().item()
    pearson = torch.corrcoef(torch.stack([preds, labels]))[0, 1].item()
    return {"mae": mae, "pearson_r": pearson}


# Training (uncomment when dataset is available)
# train_ds = ElasticityDataset(IMAGE_DIR, ANNOTATIONS_PATH, transform=train_transform)
# val_ds = ElasticityDataset(IMAGE_DIR, ANNOTATIONS_PATH, transform=val_transform)
# train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
# val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
# scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# best_mae = float("inf")
# for epoch in range(NUM_EPOCHS):
#     train_loss = train_one_epoch(model, train_loader, optimizer)
#     metrics = evaluate(model, val_loader)
#     scheduler.step()
#     print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Loss: {train_loss:.4f} | "
#           f"MAE: {metrics['mae']:.2f} | Pearson r: {metrics['pearson_r']:.4f}")
#     if metrics["mae"] < best_mae:
#         best_mae = metrics["mae"]
#         torch.save(model.state_dict(), "best_elasticity_model.pt")
#         print(f"  -> Saved best model (MAE: {best_mae:.2f})")

## ONNX Export

In [ ]:
def export_to_onnx(model, output_path="elasticity_model.onnx"):
    model.eval()
    dummy_image = torch.randn(1, 3, 224, 224).to(DEVICE)
    dummy_hc = torch.randn(1, HANDCRAFTED_DIM).to(DEVICE)

    torch.onnx.export(
        model,
        (dummy_image, dummy_hc),
        output_path,
        input_names=["image", "handcrafted_features"],
        output_names=["elasticity_score"],
        dynamic_axes={
            "image": {0: "batch"},
            "handcrafted_features": {0: "batch"},
            "elasticity_score": {0: "batch"},
        },
        opset_version=17,
    )
    print(f"Exported ONNX model to {output_path}")

    # Verify
    import onnxruntime as ort
    session = ort.InferenceSession(output_path)
    result = session.run(None, {
        "image": dummy_image.cpu().numpy(),
        "handcrafted_features": dummy_hc.cpu().numpy(),
    })
    print(f"ONNX output shape: {result[0].shape}, sample: {result[0][0][0]:.2f}")


# export_to_onnx(model)